In [2]:
import os
import time
import requests
import pandas as pd
from datetime import datetime
from urllib.parse import urlparse, parse_qs
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By

# --- [1] 카테고리 리스트 설정 ---
categories = [
    # 냉장고/김치냉장고 (CT50019441)
    {"name": "양문형 냉장고", "query": "category=CT50019441&subCategory=CT50019468"},

    # 에어컨/환기 (CT50019183)
    {"name": "2in1 에어컨", "query": "category=CT50019183&subCategory=CT50019184"},
    {"name": "벽걸이형 에어컨", "query": "category=CT50019183&subCategory=CT50019214"},
    {"name": "가정용 천장형 에어컨", "query": "category=CT50019183&subCategory=CT50172000"},
    {"name": "상업용 천장형 에어컨", "query": "category=CT50019183&subCategory=CT50019259"},
    {"name": "상업용 스탠드 에어컨", "query": "category=CT50019183&subCategory=CT50019244"},
    {"name": "이동식 에어컨", "query": "category=CT50019183&subCategory=CT50019229"},
    {"name": "창호형 에어컨", "query": "category=CT50019183&subCategory=CT50108006"},
    {"name": "가정용 환기시스템", "query": "category=CT50019183&subCategory=CT50311000"},
    {"name": "상업용 환기시스템", "query": "category=CT50019183&subCategory=CT50311001"},

    # 세탁기/건조기/의류관리기 (CT50019274)
    {"name": "드럼세탁기", "query": "category=CT50019274&subCategory=CT50019309"},
    {"name": "통돌이", "query": "category=CT50019274&subCategory=CT50019324"},
    {"name": "워시타워", "query": "category=CT50019274&subCategory=CT50109011"},
    {"name": "미니세탁기", "query": "category=CT50019274&subCategory=CT50109013"},
    {"name": "의류건조기", "query": "category=CT50019274&subCategory=CT50019275"},
    {"name": "의류관리기", "query": "category=CT50019274&subCategory=CT50019292"},
    {"name": "시스템 아이어닝", "query": "category=CT50019274&subCategory=CT50366001"},
    {"name": "슈케이스", "query": "category=CT50019274&subCategory=CT50109016"},
    {"name": "슈케어", "query": "category=CT50019274&subCategory=CT50109017"},

    # TV / 프로젝터 (CT50019021)
    {"name": "올레드 TV", "query": "category=CT50019021&subCategory=CT50019037"},
    {"name": "QNED TV", "query": "category=CT50019021&subCategory=CT50109019"},
    {"name": "울트라 HD TV", "query": "category=CT50019021&subCategory=CT50019082"},
    {"name": "일반 LED TV", "query": "category=CT50019021&subCategory=CT50019052"},
    {"name": "시네빔", "query": "category=CT50019021&subCategory=CT50019022"},
    {"name": "나노셀 TV", "query": "category=CT50019021&subCategory=CT50109023"},
    {"name": "라이프 스타일 스크린", "query": "category=CT50019021&subCategory=CT50109024"},
    {"name": "일반형(TV)", "query": "category=CT50019021&subCategory=CT50109025"},
    {"name": "프로빔 (상업용)", "query": "category=CT50019021&subCategory=CT50109026"},

    # 노트북/모니터 (CT50019563)
    {"name": "그램/노트북", "query": "category=CT50019563&subCategory=CT50019564"},
    {"name": "일체형/데스크톱", "query": "category=CT50019563&subCategory=CT50019585"},
    {"name": "PC 모니터", "query": "category=CT50019563&subCategory=CT50019602"},
    {"name": "TV 모니터", "query": "category=CT50019563&subCategory=CT50109030"},
    {"name": "태블릿", "query": "category=CT50019563&subCategory=CT50019141"},

    # 주방가전 (CT50019695)
    {"name": "식기세척기", "query": "category=CT50019695&subCategory=CT50019735"},
    {"name": "정수기", "query": "category=CT50019695&subCategory=CT50019648"},
    {"name": "전자레인지", "query": "category=CT50019695&subCategory=CT50019709"},
    {"name": "전기레인지/인덕션", "query": "category=CT50019695&subCategory=CT50019761"},
    {"name": "광파오븐레인지", "query": "category=CT50019695&subCategory=CT50019696"},
    {"name": "가스오븐레인지", "query": "category=CT50019695&subCategory=CT50019722"},
    {"name": "가스레인지", "query": "category=CT50019695&subCategory=CT50019748"},
    {"name": "맥주제조기", "query": "category=CT50019695&subCategory=CT50019774"},

    # 청소기 (CT50019339)
    {"name": "무선청소기", "query": "category=CT50019339&subCategory=CT50019400"},
    {"name": "유선청소기", "query": "category=CT50019339&subCategory=CT50019380"},
    {"name": "로봇청소기", "query": "category=CT50019339&subCategory=CT50019340"},

    # 에어케어 (CT50019872)
    {"name": "공기청정기", "query": "category=CT50019872&subCategory=CT50019873"},
    {"name": "휴대용 공기청정기", "query": "category=CT50019872&subCategory=CT50019888"},
    {"name": "제습기", "query": "category=CT50019872&subCategory=CT50019890"},
    {"name": "가습기", "query": "category=CT50019872&subCategory=CT50019904"},
    {"name": "에어로타워", "query": "category=CT50019872&subCategory=CT50111012"},
    {"name": "에어로퍼니처", "query": "category=CT50019872&subCategory=CT50111013"},
    {"name": "전자식 마스크", "query": "category=CT50019872&subCategory=CT50111014"},
    {"name": "실링팬", "query": "category=CT50019872&subCategory=CT50019889"},

    # 뷰티/의료기기 (CT50019920)
    {"name": "클렌징 기기", "query": "category=CT50019920&subCategory=CT50019921"},
    {"name": "메디헤어", "query": "category=CT50019920&subCategory=CT50019932"},
    {"name": "아이케어", "query": "category=CT50019920&subCategory=CT50111017"},
    {"name": "탄력기기", "query": "category=CT50019920&subCategory=CT50111018"},
    {"name": "흡수촉진기", "query": "category=CT50019920&subCategory=CT50111019"},
    {"name": "메디페인", "query": "category=CT50019920&subCategory=CT50111020"}
]

# --- [2] 초기 설정 ---
options = webdriver.ChromeOptions()
options.add_experimental_option("excludeSwitches", ["enable-logging"])
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

output_folder = "lg_total_crawl2"
if not os.path.exists(output_folder): os.makedirs(output_folder)

# 실시간 저장 파일명
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
csv_filename = os.path.join(output_folder, f"lg_multicat_results_{timestamp}.csv")

# --- [3] 크롤링 프로세스 시작 ---
for cat in categories:
    cat_name = cat["name"]
    cat_query = cat["query"]
    image_folder = os.path.join(output_folder, "images", cat_name)
    if not os.path.exists(image_folder): os.makedirs(image_folder)

    print(f"\n======== [{cat_name}] 수집 시작 ========")
    
    # 1단계: 해당 카테고리 1~15페이지 URL 수집
    cat_links = []
    for p in range(1, 16):
        base_url = f"https://www.lge.co.kr/support/solutions?{cat_query}&sort=update&page={p}"
        try:
            driver.get(base_url)
            time.sleep(4)
            links = driver.find_elements(By.CSS_SELECTOR, "div.solution-result-list > ul > li a")
            if not links: break # 페이지에 글이 없으면 다음 카테고리로
            
            for link in links:
                url = link.get_attribute("href")
                if url and "solutions" in url:
                    cat_links.append(url)
            print(f"   {cat_name} - {p}페이지 URL 수집 완료")
        except: break

    cat_links = list(set(cat_links))
    print(f"--- {cat_name} 총 {len(cat_links)}개 상세페이지 방문 시작 ---")

    # 2단계: 상세페이지 본문 수집 및 실시간 저장
    for idx, target_url in enumerate(cat_links):
        try:
            driver.get(target_url)
            time.sleep(5)
            
            title = driver.find_element(By.CSS_SELECTOR, "div.board-view-head > h2").text
            
            # 본문 추출 로직 (다중 셀렉터 시도)
            content_area = None
            for selector in [".content-service-view", "#viewContent > div", "#viewContent"]:
                try:
                    target = driver.find_element(By.CSS_SELECTOR, selector)
                    if target.is_displayed():
                        content_area = target
                        break
                except: continue

            body_parts = []
            img_urls = []
            if content_area:
                elements = content_area.find_elements(By.XPATH, ".//*[not(self::script or self::style or self::iframe)]")
                for el in elements:
                    txt = el.text.strip()
                    if txt and not any(txt in existing for existing in body_parts):
                        body_parts.append(txt)
                
                imgs = content_area.find_elements(By.TAG_NAME, "img")
                for img in imgs:
                    src = img.get_attribute("src")
                    if src and src.startswith("http"):
                        img_urls.append(src)
            
            # 이미지 다운로드 (서버 원본 이름 유지)
            current_saved_names = []
            for url in img_urls:
                try:
                    parsed = urlparse(url)
                    params = parse_qs(parsed.query)
                    img_name = params['fileId'][0] + ".jpg" if 'fileId' in params else os.path.basename(parsed.path)
                    if len(img_name) < 5: img_name = f"img_{idx}_{int(time.time())}.jpg"
                    
                    img_path = os.path.join(image_folder, img_name)
                    img_data = requests.get(url, timeout=10).content
                    with open(img_path, 'wb') as f: f.write(img_data)
                    current_saved_names.append(img_name)
                except: continue

            # 실시간 CSV 행 추가
            new_row = {
                '카테고리': cat_name,
                '제목': title,
                '본문': "\n".join(body_parts),
                '이미지목록': ", ".join(current_saved_names),
                'URL': target_url
            }
            
            df_row = pd.DataFrame([new_row])
            if not os.path.exists(csv_filename):
                df_row.to_csv(csv_filename, index=False, encoding="utf-8-sig")
            else:
                df_row.to_csv(csv_filename, index=False, encoding="utf-8-sig", mode='a', header=False)

            print(f"   ({idx+1}/{len(cat_links)}) {cat_name} 저장 완료: {title[:15]}...")

        except Exception as e:
            print(f"   ! 에러 발생: {target_url}")
            continue

print(f"\n✨ 모든 카테고리 수집이 완료되었습니다! 파일: {csv_filename}")
driver.quit()


======== [양문형 냉장고] 수집 시작 ========
   양문형 냉장고 - 1페이지 URL 수집 완료
   양문형 냉장고 - 2페이지 URL 수집 완료
   양문형 냉장고 - 3페이지 URL 수집 완료
   양문형 냉장고 - 4페이지 URL 수집 완료
   양문형 냉장고 - 5페이지 URL 수집 완료
   양문형 냉장고 - 6페이지 URL 수집 완료
   양문형 냉장고 - 7페이지 URL 수집 완료
   양문형 냉장고 - 8페이지 URL 수집 완료
   양문형 냉장고 - 9페이지 URL 수집 완료
   양문형 냉장고 - 10페이지 URL 수집 완료
   양문형 냉장고 - 11페이지 URL 수집 완료
   양문형 냉장고 - 12페이지 URL 수집 완료
   양문형 냉장고 - 13페이지 URL 수집 완료
   양문형 냉장고 - 14페이지 URL 수집 완료
   양문형 냉장고 - 15페이지 URL 수집 완료
--- 양문형 냉장고 총 150개 상세페이지 방문 시작 ---
   (1/150) 양문형 냉장고 저장 완료: [LG 냉장고 정수기] 디스...
   (2/150) 양문형 냉장고 저장 완료: [LG ThinQ] LG T...
   (3/150) 양문형 냉장고 저장 완료: [LG 냉장고] 디스플레이(...
   (4/150) 양문형 냉장고 저장 완료: [LG 냉장고 단열재] 제품...
   (5/150) 양문형 냉장고 저장 완료: [LG 냉장고 온도] 냉장실...
   (6/150) 양문형 냉장고 저장 완료: [LG 냉장고] 음식(식품)...
   (7/150) 양문형 냉장고 저장 완료: [LG 냉장고 소음] 시끄러...
   (8/150) 양문형 냉장고 저장 완료: [LG 냉장고 설치] 냉장고...
   (9/150) 양문형 냉장고 저장 완료: [LG 냉장고 정수기] 디스...
   (10/150) 양문형 냉장고 저장 완료: [LG ThinQ] 제품 등...
   (11/150) 양문형 냉장고 저장 완료: [LG 냉장고 정수기] 제빙...
   (12/